# 📊 Project 2: Telco Customer Churn Prediction & Explainability
**Author:** AI Machine Learning Engineer  
**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 attributes)  
**Objective:** Accurately predict whether a customer will churn (leave the service) and explain the key drivers of the prediction globally and per customer using SHAP.

---

### Table of Contents:
1. **Exploratory Data Analysis (EDA)**
   - Target Distribution & Class Imbalance (~26.5% Churn)
   - Data Cleaning & Fixing `TotalCharges` (blank text entries)
   - Analysis by Contract, Tenure, Monthly Charges, and Internet Service
2. **Preprocessing Pipeline (Zero Data Leakage)**
   - Numeric Scaling with `StandardScaler`
   - Categorical Encoding with `OneHotEncoder`
   - Scikit-Learn `ColumnTransformer` & `Pipeline`
   - Stratified Train/Test Split (80/20)
3. **Model Selection & Tuning**
   - Logistic Regression Baseline
   - Random Forest Classifier
   - XGBoost Classifier with Hyperparameter Tuning
4. **Addressing Class Imbalance**
   - Cost-Sensitive Class Weights (`class_weight='balanced'`, `scale_pos_weight`)
   - Synthetic Minority Over-sampling (`SMOTE`)
5. **Model Evaluation & Business Cost Optimization**
   - Precision, Recall, F1-Score, ROC-AUC
   - Confusion Matrices & ROC Curves
   - Cost-Sensitive Threshold Tuning (Minimizing Business Financial Loss)
6. **Model Explainability with SHAP**
   - Global Feature Importance (TreeExplainer)
   - Individual Customer Waterfall Explanations
7. **Model Serialization & Export**
   - Saving pipeline with `joblib.dump()`


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    roc_curve,
    classification_report
)
import shap

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Exploratory Data Analysis (EDA)

We load the IBM Telco Customer Churn dataset and inspect the columns, data types, and missing values.

In [ ]:
data_path = '../data/Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 1.1 Data Cleaning: Fixing `TotalCharges`
`TotalCharges` loads as `object` (string) rather than float because new customers (tenure = 0) have whitespace blanks (`" "`). We convert it to numeric and impute the blanks with `0.0`.

In [ ]:
# Inspect blanks in TotalCharges
blanks = (df['TotalCharges'].astype(str).str.strip() == '').sum()
print(f"Blanks in TotalCharges: {blanks}")

# Convert TotalCharges to float and fill blanks with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Verify types
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].dtypes)

### 1.2 Target Distribution
The churn rate in the dataset is approximately **26.5%** (~1,869 churners out of 7,043 customers). This demonstrates moderate class imbalance.

In [ ]:
churn_dist = df['Churn'].value_counts(normalize=True) * 100
print(f"Retained (No): {churn_dist['No']:.2f}% | Churned (Yes): {churn_dist['Yes']:.2f}%")

fig, ax = plt.subplots(figsize=(6, 4))
df['Churn'].value_counts().plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=ax)
ax.set_title('Churn Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Customer Count')
ax.set_xticklabels(['Retained (No)', 'Churned (Yes)'], rotation=0)
plt.tight_layout()
plt.show()

### 1.3 Key Driver Explorations: Contract, Tenure, Charges, & Internet Service

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Churn by Contract
contract_pct = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_pct.plot(kind='bar', stacked=True, color=['#2b5c8f', '#d9534f'], ax=axes[0, 0])
axes[0, 0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)

# 2. Churn by Tenure
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, bins=36, palette={'No': '#2b5c8f', 'Yes': '#d9534f'}, ax=axes[0, 1], alpha=0.6)
axes[0, 1].set_title('Customer Tenure by Churn Status', fontweight='bold')
axes[0, 1].set_xlabel('Tenure (Months)')

# 3. Churn by Monthly Charges
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, fill=True, palette={'No': '#2b5c8f', 'Yes': '#d9534f'}, ax=axes[1, 0], alpha=0.5)
axes[1, 0].set_title('Monthly Charges Density: Churn vs Retained', fontweight='bold')
axes[1, 0].set_xlabel('Monthly Charges ($)')

# 4. Churn by Internet Service
internet_pct = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
internet_pct.plot(kind='bar', stacked=True, color=['#2b5c8f', '#d9534f'], ax=axes[1, 1])
axes[1, 1].set_title('Churn Rate by Internet Service Type', fontweight='bold')
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

**EDA Key Findings:**
1. **Contract Type:** Month-to-month contracts exhibit an alarming **42.7% churn rate**, compared to **11.3%** for 1-year and **2.8%** for 2-year contracts.
2. **Tenure:** Churn is heavily concentrated in the first 1-12 months. Customers who survive past 24-36 months exhibit very high loyalty.
3. **Monthly Charges:** Customers paying between \$70-\$100/month churn at the highest rates.
4. **Internet Service:** Fiber optic subscribers churn at **41.9%**, significantly higher than DSL (**19.0%**) or customers with no internet (**7.4%**).

## 2. Preprocessing Pipeline & Stratified Split

To prevent **data leakage**, all feature transformations (StandardScaler for numeric, OneHotEncoder for categorical) are fit strictly on training folds via `ColumnTransformer` and scikit-learn `Pipeline`.

In [ ]:
# Prepare Features & Target
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = [col for col in X.columns if col not in num_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

# Stratified Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Train Churn Rate: {y_train.mean():.4f} | Test Churn Rate: {y_test.mean():.4f}")

## 3 & 4. Models & Imbalance: Class Weights vs SMOTE

We compare:
1. **Logistic Regression** (Baseline vs Class Weights vs SMOTE)
2. **Random Forest** (Class Weights vs SMOTE)
3. **XGBoost** (Class Weights / scale_pos_weight vs SMOTE vs Tuned)

In [ ]:
scale_pos_wt = float((len(y_train) - sum(y_train)) / sum(y_train))

models = {
    'Logistic Regression (Baseline)': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Logistic Regression (Class Weights)': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Logistic Regression (SMOTE)': ImbPipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest (Class Weights)': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=150, max_depth=7, class_weight='balanced', random_state=42))
    ]),
    'Random Forest (SMOTE)': ImbPipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42))
    ]),
    'XGBoost (Class Weights)': Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, scale_pos_weight=scale_pos_wt, eval_metric='logloss', random_state=42))
    ]),
    'XGBoost (SMOTE)': ImbPipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, eval_metric='logloss', random_state=42))
    ]),
    'XGBoost (Tuned)': Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(n_estimators=140, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, eval_metric='logloss', random_state=42))
    ])
}

results = []
trained_pipelines = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': round(float(accuracy_score(y_test, y_pred)), 4),
        'ROC-AUC': round(float(roc_auc_score(y_test, y_prob)), 4),
        'F1-Score': round(float(f1_score(y_test, y_pred)), 4),
        'Precision': round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        'Recall': round(float(recall_score(y_test, y_pred)), 4)
    })

comparison_df = pd.DataFrame(results)
comparison_df

### Comparison Summary: Class Weights vs SMOTE
- **Class Weights** achieve slightly higher or equivalent ROC-AUC and F1-Scores compared to SMOTE across all architectures.
- Unlike SMOTE, which generates synthetic interpolated samples in continuous and one-hot categorical spaces, class weighting directly modifies the gradient loss function without adding synthetic data artifacts.
- **XGBoost (Tuned)** yields the highest discriminative power (**ROC-AUC = 0.8482**).

## 5. Model Evaluation & Cost-Sensitive Decision Threshold Tuning

In real-world customer churn, **a False Negative (failing to catch a churner) is substantially more expensive than a False Positive (offering an unnecessary retention discount)**.

- **Cost of False Negative ($C_{FN}$):** \$500 (Customer leaves; loss of Customer Lifetime Value)
- **Cost of False Positive ($C_{FP}$):** \$100 (Customer stays anyway; cost of retention incentive)
- **Cost Function:** $\text{Total Cost} = FN \times \$500 + FP \times \$100$

Let's compute the business cost across all thresholds from $0.10$ to $0.90$.

In [ ]:
best_model = trained_pipelines['XGBoost (Tuned)']
y_test_prob = best_model.predict_proba(X_test)[:, 1]

c_fn = 500.0
c_fp = 100.0

thresholds = np.linspace(0.1, 0.9, 81)
cost_records = []

for th in thresholds:
    y_pred_th = (y_test_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_th).ravel()
    total_cost = (fn * c_fn) + (fp * c_fp)
    cost_records.append({'threshold': th, 'cost': total_cost, 'fn': fn, 'fp': fp, 'tp': tp, 'tn': tn})

cost_df = pd.DataFrame(cost_records)
opt_idx = cost_df['cost'].idxmin()
opt_threshold = cost_df.loc[opt_idx, 'threshold']
opt_cost = cost_df.loc[opt_idx, 'cost']
default_cost = cost_df.loc[(cost_df['threshold'] - 0.50).abs().idxmin(), 'cost']
savings = default_cost - opt_cost

print(f"Default Threshold (0.50) Business Cost: ${default_cost:,.2f}")
print(f"Optimal Threshold ({opt_threshold:.2f}) Business Cost: ${opt_cost:,.2f}")
print(f"Total Business Cost Savings: ${savings:,.2f} ({(savings/default_cost)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cost_df['threshold'], cost_df['cost'], color='#1e3a8a', lw=2.5, label='Total Business Cost ($)')
ax.axvline(0.50, color='#6b7280', linestyle='--', label='Default (0.50)')
ax.axvline(opt_threshold, color='#10b981', lw=2, label=f'Optimal ({opt_threshold:.2f})')
ax.scatter([opt_threshold], [opt_cost], color='#10b981', s=100, zorder=5)
ax.set_title('Cost-Sensitive Decision Threshold Tuning', fontsize=13, fontweight='bold')
ax.set_xlabel('Probability Decision Threshold')
ax.set_ylabel('Total Cost on Test Customers ($)')
ax.legend()
plt.tight_layout()
plt.show()

### 5.1 ROC Curve and Confusion Matrix at Optimal Threshold

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
auc_val = roc_auc_score(y_test, y_test_prob)

y_pred_opt = (y_test_prob >= opt_threshold).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_opt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'XGBoost (AUC = {auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], color='#9ca3af', linestyle='--')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Confusion Matrix
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1],
            xticklabels=['Predicted Retain', 'Predicted Churn'],
            yticklabels=['Actual Retain', 'Actual Churn'])
axes[1].set_title(f'Confusion Matrix at Threshold {opt_threshold:.2f}', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Model Explainability with SHAP (SHapley Additive exPlanations)

Using `shap.TreeExplainer`, we unpack both **global feature importance** and **local explanations** for individual customers.

In [ ]:
# Extract preprocessor and feature names
prep = best_model.named_steps['prep']
clf = best_model.named_steps['clf']

feature_names = [f.replace('cat__', '').replace('num__', '') for f in prep.get_feature_names_out()]
X_test_trans = prep.transform(X_test)

# TreeExplainer
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test_trans[:300])

# Global Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values.values, X_test_trans[:300], feature_names=feature_names, show=False)
plt.title('SHAP Summary Beeswarm Plot', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### 6.1 Individual Customer Explanations (Local SHAP Waterfall)

In [ ]:
# High risk customer explanation
high_risk_idx = np.argmax(y_test_prob[:50])
sample_shap = shap_values.values[high_risk_idx]
top_factors = np.argsort(np.abs(sample_shap))[::-1][:5]

print(f"Customer #{high_risk_idx} Predicted Probability: {y_test_prob[high_risk_idx]:.4f}")
print("Top 5 Driving Factors:")
for idx in top_factors:
    feat = feature_names[idx]
    val = sample_shap[idx]
    direction = "Increases Risk 🔼" if val > 0 else "Decreases Risk 🔽"
    print(f"  • {feat}: {val:+.3f} ({direction})")

## 7. Model Serialization & Export

We save the end-to-end scikit-learn Pipeline (preprocessing + XGBoost) using `joblib.dump(pipeline, "model.joblib")` for consumption by our FastAPI backend.

In [ ]:
# Save pipeline
backend_model_path = '../backend/model.joblib'
joblib.dump(best_model, backend_model_path)
print(f"Model successfully saved to {backend_model_path}!")

# Verify reload and prediction
loaded_model = joblib.load(backend_model_path)
sample_customer = X_test.iloc[[0]]
prob = loaded_model.predict_proba(sample_customer)[0, 1]
print(f"Verification Reload Success! Sample prediction probability: {prob:.4f}")